# Nguyễn Thành Trí - MSSV: 6451071079
## Phần 2 - Bài 2 (Trang 37-38) - Câu 12: Khai phá bằng FP-Growth và so sánh toàn diện với Apriori

### Đề bài:
Tìm tập phổ biến và luật kết hợp bằng thuật toán FP-Growth với cùng ngưỡng:
- `min_sup = 3%` (`0.03`)
- `min_conf = 50%` (`0.5`)
So sánh toàn diện kết quả và thời gian thực thi với thuật toán Apriori ở trên.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Phần 2 - Bài 2 (Trang 37-38) - Câu 12: So sánh Apriori và FP-Growth

import os
import time
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def load_clean_data():
    pkl_candidates = ["datasets/online_retail_clean.pkl", "../datasets/online_retail_clean.pkl", "../../datasets/online_retail_clean.pkl"]
    for p in pkl_candidates:
        if os.path.exists(p):
            return pd.read_pickle(p)
    file_path = "datasets/Online Retail.xlsx" if os.path.exists("datasets/Online Retail.xlsx") else "../datasets/Online Retail.xlsx"
    df = pd.read_excel(file_path)
    df['Description'] = df['Description'].str.strip()
    df = df.dropna(subset=['InvoiceNo'])
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    return df[~df['InvoiceNo'].str.contains('C')]

df = load_clean_data()
df_uk = df[df['Country'] == 'United Kingdom']

basket = (df_uk.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket >= 1)
if 'POSTAGE' in basket_sets.columns:
    basket_sets = basket_sets.drop(columns=['POSTAGE'])

min_sup = 0.03
min_conf = 0.5

# 1. Đo lường Apriori
t0 = time.perf_counter()
frequent_itemsets_ap = apriori(basket_sets, min_support=min_sup, use_colnames=True)
rules_ap = association_rules(frequent_itemsets_ap, metric="confidence", min_threshold=min_conf)
time_ap = time.perf_counter() - t0

# 2. Đo lường FP-Growth
t0 = time.perf_counter()
frequent_itemsets_fp = fpgrowth(basket_sets, min_support=min_sup, use_colnames=True)
rules_fp = association_rules(frequent_itemsets_fp, metric="confidence", min_threshold=min_conf)
time_fp = time.perf_counter() - t0

# Bảng so sánh
comparison_df = pd.DataFrame({
    'Chỉ số so sánh': [
        'Số tập phổ biến tìm được',
        'Số luật kết hợp sinh ra',
        'Thời gian thực thi (giây)',
        'Phương pháp tiếp cận',
        'Chi phí bộ nhớ'
    ],
    'Thuật toán Apriori': [
        len(frequent_itemsets_ap),
        len(rules_ap),
        f"{time_ap:.4f} s",
        "Level-wise (Sinh ứng viên Ck)",
        "Quét CSDL nhiều lần"
    ],
    'Thuật toán FP-Growth': [
        len(frequent_itemsets_fp),
        len(rules_fp),
        f"{time_fp:.4f} s",
        "Divide-and-Conquer (FP-Tree)",
        "Nén toàn bộ CSDL vào cây FP-Tree"
    ]
})

print("BẢNG SO SÁNH HIỆU NĂNG GIỮA APRIORI VÀ FP-GROWTH:")
display(comparison_df)

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


BẢNG SO SÁNH HIỆU NĂNG GIỮA APRIORI VÀ FP-GROWTH:


,Chỉ số so sánh,Thuật toán Apriori,Thuật toán FP-Growth
0,Số tập phổ biến tìm được,131,131
1,Số luật kết hợp sinh ra,10,10
2,Thời gian thực thi (giây),0.3993 s,0.9170 s
3,Phương pháp tiếp cận,Level-wise (Sinh ứng viên Ck),Divide-and-Conquer (FP-Tree)
4,Chi phí bộ nhớ,Quét CSDL nhiều lần,Nén toàn bộ CSDL vào cây FP-Tree


### Nhận xét & Đánh giá (Câu 12):
1. **Tính nhất quán của kết quả**: Cả hai thuật toán Apriori và FP-Growth cho ra kết quả hoàn toàn trùng khớp: đều tìm được **131** tập mục phổ biến và sinh ra đúng **10** luật kết hợp mạnh.
2. **Khác biệt về bản chất thuật toán**:
   - **Apriori**: Sử dụng nguyên lý tiên nghiệm (Apriori Property). Để tìm tập phổ biến độ dài $k$, Apriori phải sinh tập ứng viên $C_k$ từ $L_{k-1}$ và quét lại toàn bộ CSDL để đếm độ hỗ trợ. Khi số lượng giao dịch và số mặt hàng lớn, chi phí sinh ứng viên và I/O tăng mạnh.
   - **FP-Growth (Frequent Pattern Growth)**: Chỉ quét CSDL đúng 2 lần để xây dựng cấu trúc cây nén **FP-Tree**. Sau đó khai phá trực tiếp trên các cây điều kiện (conditional FP-Trees) bằng kỹ thuật chia để trị mà hoàn toàn **không cần sinh tập ứng viên**. Nhờ vậy, FP-Growth đặc biệt vượt trội khi làm việc với CSDL dày đặc hoặc khi `min_sup` nhỏ.